# PHARVO-beta: Inventory Search (Combined Filters) Test

**Objective:** Verify that an authorized user (`rafi`) can filter inventory using a combination of **Category Dropdown** (`ANALGESIC`), **Text Search** (`Napa`), and **Stock Status Filter Pill** (`All`), and verify that matching inventory records with stock levels are returned.

### Test Criteria
- **Category Filter:** `ANALGESIC` (or `Analgesic`)
- **Search Keyword:** `Napa`
- **Status Pill (Choice):** `All` (or `Low Stock`)

### Prerequisites
```bash
pip install selenium webdriver-manager
```
Ensure PHARVO frontend is running at `http://localhost:5173` and backend at `http://localhost:8000`.

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

# --- Configuration & Test Criteria ---
BASE_URL = "http://localhost:5173"
USERNAME = "rafi"
PASSWORD = "password"  # Replace with actual password

# Filter criteria: Search Text + Category + Stock Status Filter
SEARCH_KEYWORD = "Napa"         # Search keyword (medicine/brand)
CATEGORY_NAME = "ANALGESIC"     # Target category from dropdown (or 'Analgesic')
STATUS_FILTER = "All"           # Chosen status filter pill ('All', 'Low Stock', etc.)

# Step 1: Open browser and maximize window
driver = webdriver.Chrome()
driver.maximize_window()

# Set explicit wait helper (up to 10 seconds)
wait = WebDriverWait(driver, 10)

try:
    print("[INFO] Starting Combined Inventory Search & Filter Test...")

    # Step 2: Open login page and sign in
    driver.get(f"{BASE_URL}/")

    username_field = wait.until(
        EC.visibility_of_element_located((By.ID, "username"))
    )
    password_field = wait.until(
        EC.visibility_of_element_located((By.ID, "password"))
    )

    username_field.clear()
    username_field.send_keys(USERNAME)

    password_field.clear()
    password_field.send_keys(PASSWORD)

    sign_in_button = wait.until(
        EC.element_to_be_clickable((By.ID, "sign-in-btn"))
    )
    sign_in_button.click()

    # Step 3: Navigate to Medicines & Inventory
    medicines_nav = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//aside//button[contains(., 'Medicines & Inventory')]")
        )
    )
    medicines_nav.click()

    wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//header//h1[contains(text(), 'Medicines & Inventory')]")
        )
    )
    print("[INFO] Navigated to Medicines & Inventory module.")

    # Step 4: Apply Category Filter dropdown
    # Select category from the <select aria-label="Filter by category"> dropdown
    category_select_elem = wait.until(
        EC.visibility_of_element_located((By.XPATH, "//select[@aria-label='Filter by category']"))
    )
    category_select = Select(category_select_elem)

    # Find matching option (case-insensitive fallback)
    selected_cat = None
    for opt in category_select.options:
        if opt.text.strip().lower() == CATEGORY_NAME.lower():
            category_select.select_by_visible_text(opt.text)
            selected_cat = opt.text
            break

    if selected_cat:
        print(f"[INFO] Selected Category: '{selected_cat}'")
    else:
        print(f"[WARN] Category '{CATEGORY_NAME}' not found in dropdown; using default.")

    # Step 5: Apply Text Search Query
    search_input = wait.until(
        EC.visibility_of_element_located(
            (By.XPATH, "//input[@aria-label='Search medicine or brand' or @placeholder='Search medicine or brand...']")
        )
    )
    search_input.clear()
    search_input.send_keys(SEARCH_KEYWORD)
    print(f"[INFO] Entered search keyword: '{SEARCH_KEYWORD}'")

    # Step 6: Apply chosen Stock Status Filter pill (e.g. 'All' or 'Low Stock')
    status_pill = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, f"//div[@role='group' and @aria-label='Filter by stock status']//button[contains(., '{STATUS_FILTER}')]")
        )
    )
    status_pill.click()
    print(f"[INFO] Applied Stock Status Filter: '{STATUS_FILTER}'")

    # Step 7: Wait for filtered results in the inventory table
    # Give the debounced search and state a brief moment to stabilize
    time.sleep(1)

    result_rows = driver.find_elements(
        By.XPATH, "//table[contains(@class, 'med-table')]//tbody//tr[contains(@class, 'med-row-tr')]"
    )
    matching_count = len(result_rows)

    # Step 8: Verify table results against filter criteria
    if matching_count > 0:
        first_row = result_rows[0]
        med_name = first_row.find_element(By.XPATH, ".//td[1]//span[1]").text
        med_cat = first_row.find_element(By.XPATH, ".//td[2]").text
        med_stock = first_row.find_element(By.XPATH, ".//td[5]").text

        print("PASS: Combined Inventory Search & Filter executed successfully.")
        print(f"      - Total matching records: {matching_count}")
        print(f"      - Top match: '{med_name}'")
        print(f"      - Category column: '{med_cat}'")
        print(f"      - Stock level: '{med_stock}'")

        # Verify search term matches
        assert SEARCH_KEYWORD.lower() in med_name.lower(), f"Expected '{SEARCH_KEYWORD}' in '{med_name}'"
    else:
        print(f"FAIL: No inventory records matched the combined filter (Category: '{CATEGORY_NAME}', Keyword: '{SEARCH_KEYWORD}').")

except Exception as error:
    print(f"FAIL: Inventory search test encountered error: {error}")

finally:
    # Step 9: Close browser
    print("[INFO] Cleaning up and closing browser...")
    driver.quit()
